# ---------- Data loading notebook ----------

The goal of this notebook is to load and combine London bike-sharing trip data from 2024 and 2025.

Author: Artur Werys

In [ ]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as et
import numpy as np

In [17]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"

DATA_2024_DIR = DATA_DIR / "2024"
DATA_2025_DIR = DATA_DIR / "2025"

STATIONS_FILE = DATA_DIR / "stations.xml"

## ---------- Loading CSV data function ----------


In [18]:
def load_data(folder_path):
    files = list(folder_path.glob("*.csv"))
    dataframes = []

    for file in files:
        df = pd.read_csv(file)
        dataframes.append(df)

    combined_data = pd.concat(dataframes, ignore_index=True)

    return combined_data

## ---------- Loading data from choosen years ----------


In [19]:
print("--- Loading data from 2025... ---")

data_2025 = load_data(DATA_2025_DIR)

print(data_2025.head())
print("Number of trips in 2025:", len(data_2025))


print("--- Loading data from 2024... ---")

data_2024 = load_data(DATA_2024_DIR)

print(data_2024.head())
print("Number of trips in 2024:", len(data_2024))

--- Loading data from 2025... ---
      Number        Start date  Start station number  \
0  145666816  2025-01-14 23:59                  1043   
1  145666817  2025-01-14 23:59                300015   
2  145666818  2025-01-14 23:59                  1068   
3  145666819  2025-01-14 23:59                  1159   
4  145666812  2025-01-14 23:58                200048   

                      Start station          End date  End station number  \
0        Museum of London, Barbican  2025-01-15 00:13            200149.0   
1          Binfield Road, Stockwell  2025-01-15 00:04            300229.0   
2  Norton Folgate, Liverpool Street  2025-01-15 00:10              1051.0   
3         Berry Street, Clerkenwell  2025-01-15 00:10              1007.0   
4          Page Street, Westminster  2025-01-15 00:14              1058.0   

                        End station  Bike number  Bike model Total duration  \
0           Watney Street, Shadwell        55223     CLASSIC        14m 30s   
1       

### ---------- Combining datasets ----------


In [20]:
combined_trip_data = pd.concat(
    [data_2024, data_2025],
    ignore_index=True
)

### ---------- Renaming columns ----------


In [21]:
combined_trip_data = combined_trip_data.rename(columns={
    "Number": "trip_id",
    "Start date": "start_date",
    "End date": "end_date",
    "Start station number": "start_station_id",
    "Start station": "start_station_name",
    "End station number": "end_station_id",
    "End station": "end_station_name",
    "Bike number": "bike_id",
    "Bike model": "bike_model",
    "Total duration": "total_duration",
    "Total duration (ms)": "total_duration_ms",
})

combined_trip_data["total_duration_minutes"] = (
    pd.to_numeric(combined_trip_data["total_duration_ms"], errors="coerce") / 60000
)
combined_trip_data = combined_trip_data.drop(columns=["total_duration_ms"])

combined_trip_data.head()



,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_minutes
0,145207079,2024-12-14 23:59,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,0.486717
1,145207080,2024-12-14 23:59,1133,"Baylis Road, Waterloo",2024-12-15 00:26,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,27.002733
2,145207081,2024-12-14 23:59,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,18.283017
3,145207082,2024-12-14 23:59,1112,"Nutford Place, Marylebone",2024-12-15 00:12,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,13.062717
4,145207083,2024-12-14 23:59,1122,"Ashley Place, Victoria",2024-12-15 00:04,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,5.369250


### ---------- Checking for missing values in combined data ----------


In [22]:
print("Missing values in each column:")
print(combined_trip_data.isna().sum())

len_before_dropping = len(combined_trip_data)

print("Dropping rows with missing core trip data...")

combined_trip_data = combined_trip_data.dropna(
    subset=["end_date", "end_station_id", "bike_id"]
).copy()

print("Missing values in each column after dropping rows with missing core trip data:")
print(combined_trip_data.isna().sum())



Missing values in each column:
trip_id                     0
start_date                  0
start_station_id            0
start_station_name          0
end_date                  184
end_station_id            184
end_station_name          184
bike_id                     1
bike_model                  0
total_duration            184
total_duration_minutes    184
dtype: int64
Dropping rows with missing core trip data...
Missing values in each column after dropping rows with missing core trip data:
trip_id                   0
start_date                0
start_station_id          0
start_station_name        0
end_date                  0
end_station_id            0
end_station_name          0
bike_id                   0
bike_model                0
total_duration            0
total_duration_minutes    0
dtype: int64


## ---------- Loading station data from XML ----------


In [23]:
tree = et.parse(STATIONS_FILE)
root = tree.getroot()

stations_data = []

for station in root.findall("station"):

    station_data = [
        station.find("name").text.strip(),
        station.find("lat").text,
        station.find("long").text
    ]

    stations_data.append(station_data)

stations_df = pd.DataFrame(
    stations_data,
    columns=[
        "station_name",
        "latitude",
        "longitude"
    ]
)

print("--- Station data loaded from XML ---")
print(stations_df.head())
print("Number of stations:", len(stations_df))

--- Station data loaded from XML ---
                           station_name     latitude     longitude
0            River Street , Clerkenwell  51.52916347  -0.109970527
1        Phillimore Gardens, Kensington  51.49960695  -0.197574246
2  Christopher Street, Liverpool Street  51.52128377  -0.084605692
3       St. Chad's Street, King's Cross  51.53005939  -0.120973687
4         Sedding Street, Sloane Square     51.49313     -0.156876
Number of stations: 801


### ---------- Merging stations from trip data with geograpical data ----------


In [24]:
combined_trip_data = pd.merge(combined_trip_data, stations_df, left_on="start_station_name", right_on="station_name", how ="left")
combined_trip_data = combined_trip_data.rename(columns={
    "latitude": "start_lat",
    "longitude": "start_lon"
})

combined_trip_data = combined_trip_data.drop(columns=["station_name"])
combined_trip_data.head()

combined_trip_data = pd.merge(combined_trip_data, stations_df, left_on="end_station_name", right_on="station_name", how ="left")
combined_trip_data = combined_trip_data.rename(columns={
    "latitude": "end_lat",
    "longitude": "end_lon"
})

combined_trip_data = combined_trip_data.drop(columns=["station_name"])
combined_trip_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_minutes,start_lat,start_lon,end_lat,end_lon
0,145207079,2024-12-14 23:59,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,0.486717,51.50630441,-0.087262995,51.50630441,-0.087262995
1,145207080,2024-12-14 23:59,1133,"Baylis Road, Waterloo",2024-12-15 00:26,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,27.002733,51.50144456,-0.110699309,51.519265,-0.021345
2,145207081,2024-12-14 23:59,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,18.283017,51.49792478,-0.183834706,51.50035306,-0.217515071
3,145207082,2024-12-14 23:59,1112,"Nutford Place, Marylebone",2024-12-15 00:12,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,13.062717,51.5165179,-0.164393768,51.53430039,-0.1680743
4,145207083,2024-12-14 23:59,1122,"Ashley Place, Victoria",2024-12-15 00:04,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,5.369250,51.49616092,-0.140947636,51.493978,-0.127554


## ---------- Sorting rides in time for each bike ----------


In [25]:
final_trip_data = combined_trip_data.dropna(
    subset=["start_lat", "start_lon", "end_lat", "end_lon"]
).copy()



In [26]:
final_trip_data = final_trip_data.sort_values(
    ["bike_id", "start_date", "end_date"]
).copy()


## ---------- How much data was removed? ----------

In [27]:
len_after_dropping = len(final_trip_data)
dropped_rows_count = len_before_dropping - len_after_dropping

dropped_percent = dropped_rows_count / len_before_dropping * 100
remaining_percent = len_after_dropping / len_before_dropping * 100

print(f"Number of rows before dropping: {len_before_dropping:.2e}")
print(f"Number of rows after dropping: {len_after_dropping:.2e}")
print(f"Number of dropped rows: {dropped_rows_count:.2e}")

print(f"Percentage of dropped rows: {dropped_percent:.3f}%")
print(f"Percentage of remaining rows: {remaining_percent:.3f}%")



Number of rows before dropping: 1.78e+07
Number of rows after dropping: 1.74e+07
Number of dropped rows: 3.97e+05
Percentage of dropped rows: 2.228%
Percentage of remaining rows: 97.772%


## ---------- Saving final trip data as parquet ----------


In [28]:
final_trip_data["start_date"] = pd.to_datetime(
    final_trip_data["start_date"],
    format="mixed",
    dayfirst=True
)

final_trip_data["end_date"] = pd.to_datetime(
    final_trip_data["end_date"],
    format="mixed",
    dayfirst=True
)

final_trip_data["start_day_of_week"] = final_trip_data["start_date"].dt.day_name()

coordinate_columns = ["start_lat", "start_lon", "end_lat", "end_lon"]
for coordinate_column in coordinate_columns:
    final_trip_data[coordinate_column] = pd.to_numeric(
        final_trip_data[coordinate_column],
        errors="coerce"
    )

earth_radius_km = 6371.0088

start_lat_rad = np.radians(final_trip_data["start_lat"])
start_lon_rad = np.radians(final_trip_data["start_lon"])
end_lat_rad = np.radians(final_trip_data["end_lat"])
end_lon_rad = np.radians(final_trip_data["end_lon"])

delta_lat = end_lat_rad - start_lat_rad
delta_lon = end_lon_rad - start_lon_rad

haversine_a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(start_lat_rad) * np.cos(end_lat_rad) * np.sin(delta_lon / 2) ** 2
)
haversine_a = np.clip(haversine_a, 0, 1)

final_trip_data["straight_line_distance_km"] = (
    2 * earth_radius_km * np.arcsin(np.sqrt(haversine_a))
)

duration_hours = final_trip_data["total_duration_minutes"] / 60
final_trip_data["speed_km_per_h"] = np.where(
    duration_hours > 0,
    final_trip_data["straight_line_distance_km"] / duration_hours,
    np.nan
)

final_trip_data = final_trip_data.reindex(sorted(final_trip_data.columns), axis=1)

final_trip_data.to_parquet(DATA_DIR / "final_trip_data.parquet")



In [29]:
final_trip_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
4872313,2.0,CLASSIC,2024-08-01 08:26:00,51.500744,-0.202759,1115.0,"Ilchester Place, Kensington",11.269322,2024-08-01 08:12:00,Thursday,51.476885,-0.215896,200029,"Finlay Street, Fulham",2.804527,14m 55s,14.931833,141622479
4863098,2.0,CLASSIC,2024-08-01 13:43:00,51.494224,-0.236770,300037.0,"Ravenscourt Park Station, Hammersmith",8.507715,2024-08-01 13:26:00,Thursday,51.500744,-0.202759,1115,"Ilchester Place, Kensington",2.463470,17m 22s,17.373433,141631864
4809152,2.0,CLASSIC,2024-08-03 12:37:00,51.494499,-0.228188,200166.0,"Southerton Road, Hammersmith",11.829568,2024-08-03 12:34:00,Saturday,51.494224,-0.236770,300037,"Ravenscourt Park Station, Hammersmith",0.594909,3m 1s,3.017400,141687538
4757263,2.0,CLASSIC,2024-08-05 08:58:00,51.514767,-0.225787,200136.0,"BBC White City, White City",9.351433,2024-08-05 08:43:00,Monday,51.494499,-0.228188,200166,"Southerton Road, Hammersmith",2.259820,14m 29s,14.499300,141741326
3204525,2.0,CLASSIC,2024-09-05 12:00:00,51.514441,-0.087587,1201.0,"Bank of England Museum, Bank",11.196041,2024-09-05 11:51:00,Thursday,51.529537,-0.083353,3427,"Fanshaw Street, Hoxton",1.703944,9m 7s,9.131500,142640332


In [30]:
final_trip_data.head(-5)

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
4872313,2.0,CLASSIC,2024-08-01 08:26:00,51.500744,-0.202759,1115.0,"Ilchester Place, Kensington",11.269322,2024-08-01 08:12:00,Thursday,51.476885,-0.215896,200029,"Finlay Street, Fulham",2.804527,14m 55s,14.931833,141622479
4863098,2.0,CLASSIC,2024-08-01 13:43:00,51.494224,-0.236770,300037.0,"Ravenscourt Park Station, Hammersmith",8.507715,2024-08-01 13:26:00,Thursday,51.500744,-0.202759,1115,"Ilchester Place, Kensington",2.463470,17m 22s,17.373433,141631864
4809152,2.0,CLASSIC,2024-08-03 12:37:00,51.494499,-0.228188,200166.0,"Southerton Road, Hammersmith",11.829568,2024-08-03 12:34:00,Saturday,51.494224,-0.236770,300037,"Ravenscourt Park Station, Hammersmith",0.594909,3m 1s,3.017400,141687538
4757263,2.0,CLASSIC,2024-08-05 08:58:00,51.514767,-0.225787,200136.0,"BBC White City, White City",9.351433,2024-08-05 08:43:00,Monday,51.494499,-0.228188,200166,"Southerton Road, Hammersmith",2.259820,14m 29s,14.499300,141741326
3204525,2.0,CLASSIC,2024-09-05 12:00:00,51.514441,-0.087587,1201.0,"Bank of England Museum, Bank",11.196041,2024-09-05 11:51:00,Thursday,51.529537,-0.083353,3427,"Fanshaw Street, Hoxton",1.703944,9m 7s,9.131500,142640332
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12459154,99981.0,CLASSIC,2025-09-21 15:35:00,51.527152,-0.058005,200156.0,"Pott Street, Bethnal Green",0.000000,2025-09-21 15:35:00,Sunday,51.527152,-0.058005,200156,"Pott Street, Bethnal Green",0.000000,18s,0.307950,152337403
2403634,99982.0,CLASSIC,2024-02-08 10:28:00,51.510737,-0.144165,1034.0,"Bruton Street, Mayfair",2.917266,2024-02-08 10:25:00,Thursday,51.511960,-0.142783,1221,"St. George Street, Mayfair",0.166274,3m 25s,3.419800,137177798
8042011,99984.0,CLASSIC,2024-05-13 09:04:00,51.503920,-0.113426,2692.0,"Waterloo Station 2, Waterloo",0.000000,2024-05-13 09:04:00,Monday,51.503920,-0.113426,2692,"Waterloo Station 2, Waterloo",0.000000,26s,0.438517,139296798
6863953,99984.0,CLASSIC,2024-10-10 08:31:00,51.494186,-0.182671,1107.0,"Gloucester Road Station, South Kensington",3.138700,2024-10-10 08:17:00,Thursday,51.487959,-0.187405,300069,"Harcourt Terrace, West Brompton",0.766066,14m 38s,14.644267,143576758
